<a href="https://colab.research.google.com/github/michaelsteven1299/proyecto_michael-/blob/main/src/03_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 03_EDA
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno se centra en comprender profundamente los datos mediante análisis estadístico y visualización. El EDA nos permite descubrir patrones, relaciones y características importantes en los datos.

**Propósito:** Obtener insights sobre los datos y preparar el terreno para la selección de características relevantes para el modelado.

**Tareas habituales:**
- Análisis univariante de variables
- Análisis de correlaciones
- Visualización de distribuciones
- Detección de patrones y tendencias
- Análisis de series temporales si aplica
- Identificación de relaciones entre variables
- Generación de hipótesis
- Documentación de insights importantes

**Cada sección debe tener**:
- La Pregunta que se realiza sobre los datos
- El código que genera las tablas o los gráficos donde encontramos la respuesta
- El código que da respuesta a dicha pregunta
- La interpretar del resultado
- Las conclusiones

Análisis Exploratorio

IMPORTAMOS LA TABLA DESDE TRUSTED

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

CLEAN_PATH = "/content/drive/MyDrive/proyecto_oro/data/clean/"
df = pd.read_csv(CLEAN_PATH + "oro_cop_limpio.csv", index_col='DATE', parse_dates=True)

print(f"Filas: {len(df):,} | Columnas: {len(df.columns)}")
df.head()

Mounted at /content/drive
Filas: 1,377 | Columnas: 7


,oro_xauusd_Close,usd_cop_Close,oro_cop,dxy_Close,vix_Close,wti_crudo_Close,bono_10y_Close
DATE,,,,,,,
2021-01-04,182.330002,3420.250000,623614.188763,89.879997,26.969999,47.619999,0.917
2021-01-05,182.869995,3447.750000,630490.025665,89.440002,25.340000,49.930000,0.955
2021-01-06,179.899994,3442.250000,619260.753990,89.529999,25.070000,50.630001,1.042
2021-01-07,179.479996,3412.800049,612529.338183,89.830002,22.370001,50.830002,1.071
2021-01-08,173.339996,3488.239990,604651.507133,90.099998,21.559999,52.240002,1.105


In [2]:
df.describe()

,oro_xauusd_Close,usd_cop_Close,oro_cop,dxy_Close,vix_Close,wti_crudo_Close,bono_10y_Close
count,1377.000000,1377.000000,1.377000e+03,1377.000000,1377.000000,1377.000000,1377.000000
mean,230.612701,4047.143515,9.236380e+05,100.797662,19.339361,76.703907,3.452149
std,86.028380,339.005352,3.114260e+05,5.237854,5.151198,13.448961,1.131856
min,151.229996,3412.800049,5.682377e+05,89.440002,11.860000,47.619999,0.917000
25%,170.059998,3803.750000,6.898300e+05,97.779999,15.780000,67.540001,2.818000
50%,184.419998,3982.580078,7.966971e+05,101.629997,18.139999,74.559998,3.983000
75%,266.040009,4212.609863,1.102792e+06,104.430000,21.889999,82.919998,4.286000
max,495.899994,5106.000000,1.821119e+06,114.110001,52.330002,123.699997,4.988000


In [3]:
df['retorno_oro_cop'] = df['oro_cop'].pct_change()
df['retorno_oro_usd'] = df['oro_xauusd_Close'].pct_change()
df['retorno_usd_cop'] = df['usd_cop_Close'].pct_change()
df['retorno_dxy']     = df['dxy_Close'].pct_change()
df['retorno_vix']     = df['vix_Close'].pct_change()
df['retorno_wti']     = df['wti_crudo_Close'].pct_change()
df['retorno_bono_10y']= df['bono_10y_Close'].pct_change()

df[['retorno_oro_cop', 'retorno_usd_cop']].describe()

,retorno_oro_cop,retorno_usd_cop
count,1376.000000,1376.000000
mean,0.000627,0.000051
std,0.014884,0.009586
min,-0.106736,-0.043794
25%,-0.008545,-0.005834
50%,0.000602,-0.000292
75%,0.009379,0.005597
max,0.067355,0.050369


In [ ]:
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import LinearSegmentedColormap

# Paleta de color consistente para todas las graficas de este cuaderno
COLOR_PRINCIPAL = '#2a78d6'   # azul - serie principal / valores positivos
COLOR_MUTED     = '#c3c2b7'   # gris - contexto / no destacado
COLOR_CRITICO   = '#d03b3b'   # rojo - valores negativos / alerta
cmap_div = LinearSegmentedColormap.from_list('oro_diverging', [COLOR_CRITICO, '#f0efec', COLOR_PRINCIPAL])

# Titulo, unidad y si se formatea como moneda (separador de miles) por variable.
# Esto evita el problema de no saber si una grafica esta en dolares, pesos,
# puntos de indice o porcentaje.
VARS_INFO = {
    'oro_xauusd_Close': {'titulo': 'Oro internacional (XAU/USD)',        'unidad': 'USD por onza',             'moneda': True},
    'usd_cop_Close':    {'titulo': 'Tasa de cambio USD/COP',             'unidad': 'COP por USD',              'moneda': True},
    'oro_cop':          {'titulo': 'Oro en pesos colombianos',           'unidad': 'COP por onza',             'moneda': True},
    'dxy_Close':        {'titulo': 'Indice del dolar (DXY)',             'unidad': 'puntos de indice',         'moneda': False},
    'vix_Close':        {'titulo': 'Indice de volatilidad (VIX)',        'unidad': 'puntos de indice',         'moneda': False},
    'wti_crudo_Close':  {'titulo': 'Petroleo WTI',                       'unidad': 'USD por barril',           'moneda': True},
    'bono_10y_Close':   {'titulo': 'Bono del Tesoro EE.UU. a 10 anios',  'unidad': '% de rendimiento anual',   'moneda': False},
}
columnas_mercado = ['oro_xauusd_Close', 'usd_cop_Close', 'oro_cop', 'dxy_Close',
                     'vix_Close', 'wti_crudo_Close', 'bono_10y_Close']


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 14))
axes = axes.flatten()

for ax, col in zip(axes, columnas_mercado):
    info = VARS_INFO[col]
    ax.hist(df[col].dropna(), bins=40, color=COLOR_PRINCIPAL, edgecolor='white', linewidth=0.3)
    ax.set_title(info['titulo'], fontsize=11, fontweight='bold')
    ax.set_xlabel(info['unidad'], fontsize=9)
    ax.set_ylabel('Frecuencia (dias)', fontsize=9)
    if info['moneda']:
        ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[-1].axis('off')
fig.suptitle('Distribucion de las variables de mercado (2021-2026)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(len(columnas_mercado), 2, figsize=(14, 4 * len(columnas_mercado)))

for i, col in enumerate(columnas_mercado):
    info = VARS_INFO[col]

    axes[i, 0].boxplot(df[col].dropna(), vert=True)
    axes[i, 0].set_title(f"{info['titulo']} - distribucion", fontsize=10.5)
    axes[i, 0].set_ylabel(info['unidad'], fontsize=9)
    axes[i, 0].set_xticks([])
    if info['moneda']:
        axes[i, 0].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))

    axes[i, 1].plot(df.index, df[col], color=COLOR_PRINCIPAL, linewidth=0.9)
    axes[i, 1].set_title(f"{info['titulo']} - serie de tiempo", fontsize=10.5)
    axes[i, 1].set_ylabel(info['unidad'], fontsize=9)
    axes[i, 1].set_xlabel('Fecha', fontsize=9)
    if info['moneda']:
        axes[i, 1].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))

fig.suptitle('Distribucion y evolucion temporal de cada variable', fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df.index, df['oro_cop'], color=COLOR_PRINCIPAL, linewidth=1.3)
plt.title('Oro en pesos colombianos a lo largo del tiempo')
plt.ylabel('Precio del oro (COP por onza)')
plt.xlabel('Fecha')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.show()


In [ ]:
corr_movil = df['retorno_oro_usd'].rolling(60).corr(df['retorno_usd_cop'])

plt.figure(figsize=(12, 5))
plt.plot(df.index, corr_movil, color=COLOR_PRINCIPAL, linewidth=1.1)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Correlacion movil (60 dias) entre el retorno del oro (USD) y el retorno del USD/COP')
plt.ylabel('Coeficiente de correlacion (-1 a 1, sin unidades)')
plt.xlabel('Fecha')
plt.show()

print(corr_movil.describe())


In [8]:
for lag in range(1, 31):
    df[f'usd_cop_lag_{lag}'] = df['retorno_usd_cop'].shift(lag)

df.filter(like='usd_cop_lag').head()

,usd_cop_lag_1,usd_cop_lag_2,usd_cop_lag_3,usd_cop_lag_4,usd_cop_lag_5,usd_cop_lag_6,usd_cop_lag_7,usd_cop_lag_8,usd_cop_lag_9,usd_cop_lag_10,...,usd_cop_lag_21,usd_cop_lag_22,usd_cop_lag_23,usd_cop_lag_24,usd_cop_lag_25,usd_cop_lag_26,usd_cop_lag_27,usd_cop_lag_28,usd_cop_lag_29,usd_cop_lag_30
DATE,,,,,,,,,,,,,,,,,,,,,
2021-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-06,0.008040,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-07,-0.001595,0.008040,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-08,-0.008555,-0.001595,0.00804,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
target = df['retorno_oro_cop']
correlaciones = {lag: df[f'usd_cop_lag_{lag}'].corr(target) for lag in range(1, 31)}
corr_lags = pd.Series(correlaciones)

print(corr_lags.sort_values(key=abs, ascending=False))

fig, ax = plt.subplots(figsize=(14, 6))
colores = [COLOR_PRINCIPAL if lag % 5 == 0 else COLOR_MUTED for lag in corr_lags.index]
ax.bar(corr_lags.index, corr_lags.values, color=colores)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Rezago del retorno USD/COP (dias habiles)')
ax.set_ylabel('Correlacion con el retorno del oro en COP (-1 a 1)')
ax.set_title('Correlacion por rezago del USD/COP - en azul, los multiplos de 5 dias')
plt.show()


In [ ]:
from scipy.stats import f_oneway

df['dia_semana'] = df.index.day_name()
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
df['dia_semana'] = pd.Categorical(df['dia_semana'], categories=orden_dias, ordered=True)

resumen_dia = df.groupby('dia_semana', observed=True)['retorno_usd_cop'].agg(['mean', 'std', 'count'])
print(resumen_dia)

grupos = [df[df['dia_semana'] == d]['retorno_usd_cop'].dropna() for d in orden_dias]
f_stat, p_valor = f_oneway(*grupos)
print(f"\nF-statistic: {f_stat:.4f} | p-valor: {p_valor:.10f}")

medias_pct = df.groupby('dia_semana', observed=True)['retorno_usd_cop'].mean() * 100
colores_dia = [COLOR_PRINCIPAL if v >= 0 else COLOR_CRITICO for v in medias_pct.values]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes'], medias_pct.values, color=colores_dia)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Retorno promedio del USD/COP (%)')
ax.set_title('Retorno promedio del USD/COP por dia de la semana')
plt.show()


In [ ]:
plt.figure(figsize=(20, 16))
sns.heatmap(df.corr(numeric_only=True), cmap=cmap_div, center=0, vmin=-1, vmax=1,
            cbar_kws={'label': 'Coeficiente de correlacion (-1 a 1)'})
plt.title('Matriz de correlacion: variables de mercado, retornos y rezagos del USD/COP', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()


In [12]:
otras = ['retorno_dxy', 'retorno_vix', 'retorno_wti', 'retorno_bono_10y']
correlacion_otras = df[otras].corrwith(df['retorno_oro_cop'])
print("Correlación de retornos (análisis exploratorio complementario):")
print(correlacion_otras.sort_values(key=abs, ascending=False))

Correlación de retornos (análisis exploratorio complementario):
retorno_dxy        -0.309132
retorno_bono_10y   -0.200819
retorno_vix        -0.091293
retorno_wti         0.070398
dtype: float64


In [13]:
# Reconstruir con ffill(limit=3), igual al pipeline original que generó trusted
df_test = df[['oro_xauusd_Close', 'usd_cop_Close']].copy()
df_test = df_test.asfreq('D')  # aseguramos frecuencia diaria para que ffill tenga huecos que rellenar
df_test = df_test.ffill(limit=3)
df_test = df_test[df_test.index.dayofweek < 5]

df_test['oro_cop'] = df_test['oro_xauusd_Close'] * df_test['usd_cop_Close']
df_test['retorno_oro_cop'] = df_test['oro_cop'].pct_change()
df_test['retorno_usd_cop'] = df_test['usd_cop_Close'].pct_change()

for lag in range(1, 31):
    df_test[f'usd_cop_lag_{lag}'] = df_test['retorno_usd_cop'].shift(lag)

target = df_test['retorno_oro_cop']
correlaciones_test = {lag: df_test[f'usd_cop_lag_{lag}'].corr(target) for lag in range(1, 31)}
corr_test = pd.Series(correlaciones_test)

print(f"Filas con ffill: {len(df_test)}")
print(corr_test.sort_values(key=abs, ascending=False))

Filas con ffill: 1432
5     0.129394
30    0.119831
20    0.116629
10    0.107631
26   -0.083056
21   -0.078787
4    -0.066127
16   -0.065450
8     0.057137
12   -0.056343
19   -0.048202
7    -0.047287
6    -0.043112
14   -0.034425
24   -0.033637
15    0.031068
9    -0.030322
2    -0.030044
18    0.029384
17   -0.025411
22    0.021842
23   -0.019174
27   -0.018287
3    -0.017031
1    -0.014777
29   -0.012642
28    0.010155
13    0.007399
25   -0.005034
11    0.003908
dtype: float64


/tmp/ipykernel_505/2205600088.py:8: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_test['retorno_oro_cop'] = df_test['oro_cop'].pct_change()
/tmp/ipykernel_505/2205600088.py:9: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_test['retorno_usd_cop'] = df_test['usd_cop_Close'].pct_change()


In [14]:
# Con base en la Celda 9 (sin ffill, tu resultado más riguroso):
# lag 5, 8, 10 (positivos) y lag 16 (negativo) fueron los más destacados
lags_seleccionados = [5, 8, 10, 16]

cols_seleccionadas = [f'usd_cop_lag_{lag}' for lag in lags_seleccionados]
df[cols_seleccionadas].head()

,usd_cop_lag_5,usd_cop_lag_8,usd_cop_lag_10,usd_cop_lag_16
DATE,,,,
2021-01-04,NaN,NaN,NaN,NaN
2021-01-05,NaN,NaN,NaN,NaN
2021-01-06,NaN,NaN,NaN,NaN
2021-01-07,NaN,NaN,NaN,NaN
2021-01-08,NaN,NaN,NaN,NaN


In [15]:
df['dia_semana_num'] = df.index.dayofweek  # 0=lunes, 4=viernes
dummies_dia = pd.get_dummies(df['dia_semana_num'], prefix='dia', drop_first=True)
df = pd.concat([df, dummies_dia], axis=1)

print(dummies_dia.columns.tolist())

['dia_1', 'dia_2', 'dia_3', 'dia_4']


In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

features = cols_seleccionadas + dummies_dia.columns.tolist()
df_modelo = df.dropna(subset=features + ['retorno_oro_cop'])

X = df_modelo[features]
y = df_modelo['retorno_oro_cop']

split = int(len(df_modelo) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

modelo = LinearRegression()
modelo.fit(X_train, y_train)
pred = modelo.predict(X_test)

rmse_modelo = np.sqrt(mean_squared_error(y_test, pred))
rmse_persistencia = np.sqrt(mean_squared_error(y_test, np.zeros(len(y_test))))

print(f"RMSE Modelo reducido (4 lags + día semana): {rmse_modelo:.5f}")
print(f"RMSE referencia (retorno = 0):                {rmse_persistencia:.5f}")
print(f"\n¿Mejoró? {'SÍ' if rmse_modelo < rmse_persistencia else 'NO'}")

RMSE Modelo reducido (4 lags + día semana): 0.01897
RMSE referencia (retorno = 0):                0.01928

¿Mejoró? SÍ


In [17]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=6)
rmses_modelo = []
rmses_referencia = []

for train_idx, test_idx in tscv.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    m = LinearRegression()
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)

    rmses_modelo.append(np.sqrt(mean_squared_error(y_te, pred)))
    rmses_referencia.append(np.sqrt(mean_squared_error(y_te, np.zeros(len(y_te)))))

print("RMSE por ventana (modelo reducido):", [round(r, 5) for r in rmses_modelo])
print("RMSE por ventana (referencia):      ", [round(r, 5) for r in rmses_referencia])
print(f"\nVentanas donde el modelo ganó: {sum(m < r for m, r in zip(rmses_modelo, rmses_referencia))} de {len(rmses_modelo)}")

RMSE por ventana (modelo reducido): [np.float64(0.01323), np.float64(0.01582), np.float64(0.01238), np.float64(0.01288), np.float64(0.01414), np.float64(0.02124)]
RMSE por ventana (referencia):       [np.float64(0.01318), np.float64(0.01602), np.float64(0.01247), np.float64(0.0132), np.float64(0.0147), np.float64(0.02161)]

Ventanas donde el modelo ganó: 5 de 6


In [18]:
df['retorno_dxy_lag1']      = df['retorno_dxy'].shift(1)
df['retorno_vix_lag1']      = df['retorno_vix'].shift(1)
df['retorno_wti_lag1']      = df['retorno_wti'].shift(1)
df['retorno_bono_10y_lag1'] = df['retorno_bono_10y'].shift(1)

df[['retorno_dxy_lag1', 'retorno_vix_lag1', 'retorno_wti_lag1', 'retorno_bono_10y_lag1']].describe()

,retorno_dxy_lag1,retorno_vix_lag1,retorno_wti_lag1,retorno_bono_10y_lag1
count,1375.000000,1375.000000,1375.000000,1375.000000
mean,0.000095,0.002801,0.000608,0.001359
std,0.004395,0.082088,0.025160,0.021108
min,-0.021167,-0.357539,-0.164143,-0.099088
25%,-0.002441,-0.041143,-0.013658,-0.010588
50%,0.000190,-0.007466,0.001945,0.000705
75%,0.002723,0.036379,0.015032,0.012354
max,0.016525,0.740391,0.122084,0.092873


In [19]:
nuevas_vars = ['retorno_dxy_lag1', 'retorno_vix_lag1', 'retorno_wti_lag1', 'retorno_bono_10y_lag1']
corr_nuevas = df[nuevas_vars].corrwith(df['retorno_oro_cop'])
print(corr_nuevas.sort_values(key=abs, ascending=False))

retorno_dxy_lag1         0.195943
retorno_vix_lag1         0.102065
retorno_wti_lag1        -0.087671
retorno_bono_10y_lag1    0.076997
dtype: float64


In [20]:
features_ampliado = cols_seleccionadas + dummies_dia.columns.tolist() + nuevas_vars
df_modelo2 = df.dropna(subset=features_ampliado + ['retorno_oro_cop'])

X2 = df_modelo2[features_ampliado]
y2 = df_modelo2['retorno_oro_cop']

tscv = TimeSeriesSplit(n_splits=6)
rmses_ampliado = []
rmses_ref2 = []

for train_idx, test_idx in tscv.split(X2):
    X_tr, X_te = X2.iloc[train_idx], X2.iloc[test_idx]
    y_tr, y_te = y2.iloc[train_idx], y2.iloc[test_idx]

    m = LinearRegression()
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)

    rmses_ampliado.append(np.sqrt(mean_squared_error(y_te, pred)))
    rmses_ref2.append(np.sqrt(mean_squared_error(y_te, np.zeros(len(y_te)))))

print("RMSE modelo ampliado (8 vars):", [round(r, 5) for r in rmses_ampliado])
print("RMSE referencia:               ", [round(r, 5) for r in rmses_ref2])
print(f"\nVentanas donde el modelo ampliado ganó: {sum(m < r for m, r in zip(rmses_ampliado, rmses_ref2))} de {len(rmses_ampliado)}")

RMSE modelo ampliado (8 vars): [np.float64(0.01312), np.float64(0.01517), np.float64(0.01175), np.float64(0.01216), np.float64(0.01388), np.float64(0.0213)]
RMSE referencia:                [np.float64(0.01318), np.float64(0.01602), np.float64(0.01247), np.float64(0.0132), np.float64(0.0147), np.float64(0.02161)]

Ventanas donde el modelo ampliado ganó: 6 de 6


In [21]:
print("Comparación directa por ventana:")
print(f"{'Ventana':<10}{'Reducido (4)':<15}{'Ampliado (8)':<15}{'Referencia':<12}")
for i in range(6):
    print(f"{i+1:<10}{rmses_modelo[i]:<15.5f}{rmses_ampliado[i]:<15.5f}{rmses_referencia[i]:<12.5f}")

Comparación directa por ventana:
Ventana   Reducido (4)   Ampliado (8)   Referencia  
1         0.01323        0.01312        0.01318     
2         0.01582        0.01517        0.01602     
3         0.01238        0.01175        0.01247     
4         0.01288        0.01216        0.01320     
5         0.01414        0.01388        0.01470     
6         0.02124        0.02130        0.02161     


In [ ]:
features_ampliado = cols_seleccionadas + dummies_dia.columns.tolist() + nuevas_vars

matriz_features = df[features_ampliado].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_features, cmap=cmap_div, center=0, vmin=-1, vmax=1, annot=True, fmt='.2f',
            cbar_kws={'label': 'Coeficiente de correlacion (-1 a 1)'})
plt.title('Correlacion entre las variables predictoras del modelo (multicolinealidad)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [23]:
# La pregunta clave: ¿el retorno de usd_cop y el retorno de dxy se mueven parecido?
corr_dxy_usdcop = df['retorno_usd_cop'].corr(df['retorno_dxy'])
print(f"Correlación entre retorno_usd_cop y retorno_dxy (mismo día): {corr_dxy_usdcop:.4f}")

# Y entre sus versiones rezagadas usadas en el modelo
corr_lags = df['usd_cop_lag_5'].corr(df['retorno_dxy_lag1'])
print(f"Correlación entre usd_cop_lag_5 y retorno_dxy_lag1: {corr_lags:.4f}")

Correlación entre retorno_usd_cop y retorno_dxy (mismo día): 0.0286
Correlación entre usd_cop_lag_5 y retorno_dxy_lag1: -0.0296


In [24]:
UMBRAL = 0.8

pares_problematicos = []
for i, var1 in enumerate(features_ampliado):
    for var2 in features_ampliado[i+1:]:
        corr = df[var1].corr(df[var2])
        if abs(corr) > UMBRAL:
            pares_problematicos.append((var1, var2, corr))

if pares_problematicos:
    print("⚠ Pares con posible multicolinealidad (>0.8):")
    for v1, v2, c in pares_problematicos:
        print(f"  {v1} - {v2}: {c:.3f}")
else:
    print("✓ Ningún par de variables supera el umbral de 0.8 — no hay multicolinealidad severa")

✓ Ningún par de variables supera el umbral de 0.8 — no hay multicolinealidad severa
